# CAWOT-CM V0 — Kaggle end-to-end

**Setup (Kaggle UI)**
- Settings → Accelerator: **GPU P100**
- Settings → Internet: **On**
- Add Data → friend's image dataset (e.g. `vnhtbo/pab-eccv26-track4-train-webp-part-01-05`)
- Add Data → friend's qproxy dataset (the folder from Drive uploaded as Kaggle dataset)

What this notebook does:
1. Clone repo + install deps
2. Patch `config.yaml` with the Kaggle paths visible in `/kaggle/input/`
3. Run V0 end-to-end (cluster → select Random + V0 → fine-tune CLIP-B/16 on each → eval)
4. Print summary table

## 1. Sanity check env

In [ ]:
!nvidia-smi -L
!ls /kaggle/input/

## 2. Get the code

In [ ]:
import os
if not os.path.exists("/kaggle/working/cawot-cm"):
    !git clone https://github.com/HohoHocCode/cawot-cm.git /kaggle/working/cawot-cm
%cd /kaggle/working/cawot-cm

## 3. Install dependencies (~2 min)

In [ ]:
!pip install -q open_clip_torch faiss-gpu-cu12 pyarrow einops

## 4. Configure paths

Adjust `QPROXY_DIR` and `IMAGE_DATASET_PREFIX` based on what shows up in `/kaggle/input/`.

If your image dataset slug differs from the one baked into friend's manifest (`vnhtbo/pab-eccv26-track4-train-webp-part-01-05`), set `PATH_REMAP` accordingly. Otherwise leave it empty.

In [ ]:
import yaml

# === Edit these 2-3 lines ===
QPROXY_DIR = "/kaggle/input/cawot-qproxy"   # folder containing manifest + embeddings
PATH_REMAP = {}   # leave empty unless your image dataset slug differs

# Example PATH_REMAP if friend's manifest path differs from your Kaggle slug:
# PATH_REMAP = {
#     "/kaggle/input/datasets/vnhtbo/pab-eccv26-track4-train-webp-part-01-05":
#         "/kaggle/input/your-image-dataset-slug",
# }

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["qproxy"]["manifest_path"] = f"{QPROXY_DIR}/image_subset_manifest.parquet"
cfg["qproxy"]["embeddings_path"] = f"{QPROXY_DIR}/image_subset_embeddings.npy"
cfg["qproxy"]["path_remap"] = PATH_REMAP

# Kaggle has 4 CPU cores — keep workers modest
cfg["data"]["num_workers"] = 2

# Output paths inside /kaggle/working (persisted as notebook outputs)
cfg["cluster"]["output_path"] = "/kaggle/working/outputs/clusters.npz"
cfg["coreset"]["output_dir"] = "/kaggle/working/outputs"
cfg["train"]["output_dir"] = "/kaggle/working/outputs/checkpoints"
cfg["eval"]["output_dir"] = "/kaggle/working/outputs/eval"

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(yaml.dump(cfg, sort_keys=False))

## 5. Sanity check that paths resolve

Loads the manifest, picks a few rows, and verifies the image files exist on disk. Catches `path_remap` mistakes early.

In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working/cawot-cm")
from src.data import load_manifest

manifest = load_manifest(
    cfg["qproxy"]["manifest_path"],
    path_remap=cfg["qproxy"].get("path_remap") or None,
)
print(f"manifest rows: {len(manifest)}")
print(manifest[["image_id", "image_path"]].head(3).to_string(index=False))

n_ok = sum(os.path.exists(p) for p in manifest["image_path"].head(20))
print(f"\nfirst 20 rows: {n_ok}/20 image files exist on disk")
assert n_ok > 15, "path_remap looks wrong — check that the prefix matches your Kaggle dataset slug"

## 6. Run end-to-end

On Kaggle P100 with the ~47K subset:
- Cluster + select: ~1 min
- Train each coreset (3 epochs, ~9K samples): ~15-25 min
- Eval each: ~2 min
- Total: ~45-60 min for Random + V0 + zero-shot

In [ ]:
!python scripts/run_v0.py --config config.yaml

## 7. Results

In [ ]:
import json, pandas as pd
with open("/kaggle/working/outputs/eval/summary.json") as f:
    summary = json.load(f)
df = pd.DataFrame(summary).T
df

## 8. What to look for

- **`mean_R@1`**: average of text→image and image→text recall at 1 over the held-out 2K val pairs.
- **V0 > Random > zero-shot** is the expected ordering. Typical gap: V0 beats Random by 0.5–1.5%.
- If V0 ≤ Random: bug in selection or K is wrong scale. Sanity check before V1.
- If both methods barely beat zero-shot: training collapsed (LR too high, bad augmentation, etc.).

Save the `summary.json` and `outputs/checkpoints/` to commit them to the notebook output for reproducibility.